# Proyecto - Adquisición de Datos
---

- Saul Baltazar
- Jean Luka Terrazo
- Jose Wong


Este notebook contiene el script utilizado para la fase de adquisición de datos para el desarrollo del proyecto.

Se utilizó la página de [Computrabajo Perú](https://pe.computrabajo.com/), fijando la búsqueda en cargo=`Analista` y lugar=`Lima`.

Esta búsqueda arrojó 4226 ofertas de trabajo almacenados en 212 páginas, los cuales se encuentran en 3 archivos csv. La razón para dividir el dataset en 3 partes es para principalmente optimizar el tiempo de scrapeo, puesto que el script permite empezar desde una página determinada. Esto contribuyó a que cada integrante del grupo pueda scrapear un tercio de las ofertas, evitando posibles fallas respecto al tiempo de ejecución del notebook.

Como la cantidad de ofertas cambia dependiendo del momento de ejecución, se decidió que la parte 3 del dataset consistiría únicamente desde donde termina la parte 2 hasta la página 200.


In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import re

In [ ]:
DOMINIO = "https://pe.computrabajo.com"
BASE_URL = f"{DOMINIO}/trabajo-de-analista-en-lima"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [ ]:
def extraer_datos_nivel_1(bloque_oferta, dominio_base):
    """ Extrae Puesto, URL y Ubicación del bloque de lista, asegurando la URL. """
    datos_oferta = {
        'url_detalle': None,
        'cargo_especifico': None,
        'empresa_nombre': None,
        'ubicacion_provincia': None
    }

    try:
        # Puesto y URL
        titulo_h2 = bloque_oferta.find('h2', class_='fs18 fwB prB')
        if titulo_h2:
            link_tag = titulo_h2.find('a')
            if link_tag:
                href = link_tag.get('href')
                datos_oferta['url_detalle'] = urljoin(dominio_base, href.split('#')[0])
                datos_oferta['cargo_especifico'] = link_tag.get_text(strip=True)

        # Empresa
        empresa_tag = bloque_oferta.find('p', class_='dFlex vm_fx fs16 fc_base mt5')
        if empresa_tag:
            datos_oferta['empresa_nombre'] = empresa_tag.get_text(strip=True)

        # Ubicación
        location_tag = bloque_oferta.find('p', class_='fs16 fc_base mt5')
        if location_tag:
            location_span = location_tag.find('span', class_='mr10')
            if location_span:
                datos_oferta['ubicacion_provincia'] = location_span.get_text(strip=True)

    except Exception as e:
        print(f"Error Nivel I: {e}")

    return datos_oferta

In [ ]:
def extraer_datos_nivel_2(html_content):
    """ Extrae Salario, Experiencia, Contrato, Jornada, Menciones Tech, Educación Mínima y MODALIDAD,
        buscando directamente los tags de detalle."""

    oferta_detalle = {
        'salario_mensual_base': None,
        'experiencia_requerida': None,
        'menciones_tecnologia': 0,
        'tipo_contrato': None,
        'jornada': None,
        'educacion_minima': None,
        'modalidad': 'Presencial' # Valor por defecto
    }

    try:
        SOUP_DETAIL = BeautifulSoup(html_content, 'html.parser')
        full_text = SOUP_DETAIL.get_text().lower()

        tags = SOUP_DETAIL.find_all('span', class_='tag base mb10')

        for tag in tags:
            tag_text = tag.get_text(strip=True)
            tag_text_lower = tag_text.lower()

            # A. Salario (Numérico) o "A convenir"
            salario_match = re.search(r'S\/\.\s*([\d\.,]+)', tag_text)
            if salario_match:
                valor_str = salario_match.group(1).replace('.', '').replace(',', '.')
                oferta_detalle['salario_mensual_base'] = float(valor_str)
                continue

            # B. Jornada (Tiempo completo/parcial)
            if 'tiempo' in tag_text_lower:
                if 'completo' in tag_text_lower:
                    oferta_detalle['jornada'] = 'Completa'
                elif 'parcial' in tag_text_lower:
                    oferta_detalle['jornada'] = 'Parcial'
                continue

            # C. Modalidad (Presencial/Remoto/Híbrido)
            if 'presencial y remoto' in tag_text_lower:
                oferta_detalle['modalidad'] = 'Híbrido'
                continue
            elif 'remoto' in tag_text_lower:
                oferta_detalle['modalidad'] = 'Remoto'
                continue

            # D. Tipo de Contrato (Si no es salario, jornada, modalidad o "A convenir")
            if tag_text_lower != 'a convenir':
                oferta_detalle['tipo_contrato'] = tag_text
                continue

        # Educación Mínima
        edu_tag_li = SOUP_DETAIL.find('li', string=lambda t: t and 'educación mínima' in t.lower())
        if edu_tag_li:
            edu_text = edu_tag_li.get_text().lower()
            edu_match = re.search(r'educación mínima:\s*([a-záéíóú\s]+)', edu_text)
            if edu_match:
                oferta_detalle['educacion_minima'] = edu_match.group(1).strip()

        # Experiencia Requerida (Prioridad: Estructurada)
        ul_requerimientos = SOUP_DETAIL.find('p', class_='fwB fs18 mtB mb10', string=lambda t: t and 'requerimientos' in t.lower())
        if ul_requerimientos:
            ul_tag = ul_requerimientos.find_next_sibling('ul', class_='disc mbB')
            if ul_tag:
                requerimientos_text = ul_tag.get_text().lower()
                exp_match_req = re.search(r'(\d+)\s+año[s]?\s+de\s+experiencia', requerimientos_text)
                if exp_match_req:
                    oferta_detalle['experiencia_requerida'] = int(exp_match_req.group(1))

        # Menciones Tecnológicas
        tecnologias = ['python', 'sql', 'excel', 'power bi', 'tableau', 'big data', 'ia', 'machine learning', 'sas', 'r']
        for tech in tecnologias:
            if tech in full_text:
                oferta_detalle['menciones_tecnologia'] += 1

    except Exception as e:
        print(f"Error Nivel II: {e}")

    return oferta_detalle

In [ ]:
def ejecutar_scrapeo_por_rango(pagina_inicio, pagina_fin, nombre_archivo_salida):
    """
    Ejecuta el pipeline de scraping en el rango de páginas especificado.
    """
    LISTA_DATOS_FINAL = []
    conteo_registros = 0
    total_paginas = pagina_fin - pagina_inicio + 1

    print("-" * 50)
    print(f"--- INICIANDO SCRAPING: RANGO DE PÁGINAS {pagina_inicio} A {pagina_fin} ---")
    print(f"Total de páginas a procesar: {total_paginas}")

    for i in range(pagina_inicio, pagina_fin + 1):
        URL_PAGINA = f"{BASE_URL}?p={i}"

        try:
            print(f"\nProcesando página {i}/{pagina_fin}: {URL_PAGINA}")

            # 1. SOLICITUD Nivel I (Página de resultados)
            response_n1 = requests.get(URL_PAGINA, headers=HEADERS)
            response_n1.raise_for_status()
            SOUP_N1 = BeautifulSoup(response_n1.content, 'html.parser')

            BLOQUES_OFERTAS = SOUP_N1.find_all('article', class_='box_offer')

            if not BLOQUES_OFERTAS:
                print(f"Página {i} vacía. Fin de los resultados de la búsqueda.")
                break

            # 2. BUCLE: Iterar sobre cada bloque de oferta
            for index, bloque in enumerate(BLOQUES_OFERTAS, 1):
                registro_final = {}

                datos_n1 = extraer_datos_nivel_1(bloque, DOMINIO)
                registro_final.update(datos_n1)
                url_de_detalle = datos_n1['url_detalle']

                if url_de_detalle:
                    # B. Extracción Nivel II (Detalle)
                    try:
                        response_n2 = requests.get(url_de_detalle, headers=HEADERS, timeout=10)
                        response_n2.raise_for_status()

                        datos_n2 = extraer_datos_nivel_2(response_n2.content)
                        registro_final.update(datos_n2)

                        LISTA_DATOS_FINAL.append(registro_final)
                        conteo_registros += 1

                    except requests.exceptions.RequestException as e:
                        print(f"Error Nivel II en oferta {index}: {e}")

                    time.sleep(2)

            print(f"Página {i} procesada. Registros extraídos hasta ahora: {conteo_registros}")
            time.sleep(10)

        except requests.exceptions.RequestException as e:
            print(f"Error de conexión en página {i}: {e}. Esperando 30s y saltando a la siguiente página...")
            time.sleep(30)
            continue
        except Exception as e:
            print(f"Error inesperado en página {i}: {e}. Saltando a la siguiente página...")
            time.sleep(10)
            continue

    # --- EXPORTACIÓN FINAL ---
    df_final = pd.DataFrame(LISTA_DATOS_FINAL)

    print("-" * 50)
    print(f"Proceso completado. Total de registros válidos extraídos: {len(df_final)}")
    df_final.to_csv(nombre_archivo_salida, index=False, encoding='utf-8')
    print(f"Archivo '{nombre_archivo_salida}' exportado con éxito.")
    print("-" * 50)

In [ ]:
"""
Tenemos 4226 ofertas de trabajo de analista en Lima. 20 ofertas por página -> 212 páginas
Tenemos una pausa de 2 segundos entre cada oferta, más una pausa de 10 segundos por página.

En total, si uno solo ejecuta el script, tendría un tiempo de ~ 3 horas.
Mas viable que dividamos la carga de trabajo en 3 scripts con una hora de ejecución por cada uno.
Unir los scripts sería data integration? Si es así, entregamos los scripts por separado.

En resumen, así queda las ejecuciones:
a) 1 - 71
b) 72 - 142
c) 143 - 212
"""

PAGINA_INICIO_GRUPO = 72
PAGINA_FIN_GRUPO = 142
NOMBRE_ARCHIVO_GRUPO = "dataset_analista_lima_parte_2.csv"

ejecutar_scrapeo_por_rango(PAGINA_INICIO_GRUPO, PAGINA_FIN_GRUPO, NOMBRE_ARCHIVO_GRUPO)

--------------------------------------------------
--- INICIANDO SCRAPING: RANGO DE PÁGINAS 72 A 142 ---
Total de páginas a procesar: 71

Procesando página 72/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=72
Página 72 procesada. Registros extraídos hasta ahora: 20

Procesando página 73/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=73
Página 73 procesada. Registros extraídos hasta ahora: 40

Procesando página 74/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=74
Página 74 procesada. Registros extraídos hasta ahora: 60

Procesando página 75/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=75
Página 75 procesada. Registros extraídos hasta ahora: 80

Procesando página 76/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=76
Página 76 procesada. Registros extraídos hasta ahora: 100

Procesando página 77/142: https://pe.computrabajo.com/trabajo-de-analista-en-lima?p=77
Página 77 procesada. Registros extraídos hasta ah

In [ ]:
# De la prueba

df_display

,cargo_especifico,ubicacion_provincia,modalidad,salario_mensual_base,experiencia_requerida,educacion_minima,tipo_contrato,jornada,menciones_tecnologia,url_detalle
0,Analista GTR Ventas Portabilidad,"Lima, Lima",Presencial,NaN,1.0,técnico,Contrato por Inicio o Incremento de Actividad,Completa,5,https://pe.computrabajo.com/ofertas-de-trabajo...
1,Analista Contable,"Santa Anita, Lima",Presencial,2000.0,2.0,universitario,Contrato por Obra Determinada o Servicio Espec...,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
2,Analista de Almacén MP / Temporal (Suplencia ...,"Ate, Lima",Presencial,NaN,2.0,universitario,Contrato de Suplencia,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
3,Analista de Control de CalIdad,"Ate, Lima",Presencial,NaN,3.0,universitario,Contrato por Obra Determinada o Servicio Espec...,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
4,Analista contable,"San Luis, Lima",Presencial,2700.0,2.0,universitario,Contrato por Obra Determinada o Servicio Espec...,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
5,Analista de propuestas,"Magdalena Del Mar, Lima",Presencial,NaN,2.0,universitario,Contrato por Inicio o Incremento de Actividad,Completa,3,https://pe.computrabajo.com/ofertas-de-trabajo...
6,Analista Programador de Sistemas .NET Presenci...,"Pisco, Ica",Presencial,NaN,2.0,técnico,Contrato por Obra Determinada o Servicio Espec...,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
7,Analista de Créditos y Cobranzas,"Villa Maria Del Triunfo, Lima",Presencial,NaN,2.0,técnico,Contrato por Necesidades del Mercado,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
8,Analista de Gestión del Talento,"Santiago De Surco, Lima",Híbrido,NaN,NaN,técnico,Contrato por Inicio o Incremento de Actividad,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...
9,analista de calidad,"Lima, Lima",Híbrido,NaN,1.0,universitario,Contrato por Obra Determinada o Servicio Espec...,Completa,4,https://pe.computrabajo.com/ofertas-de-trabajo...


## Unión de los archivos

Una vez que teniamos listos los 3 archivos csv que compondrían el dataset, se unirán en un único `dataset_analista_lima.csv`

In [2]:
import os

In [3]:
files = [
    "/content/dataset_analista_lima_parte_1.csv",
    "/content/dataset_analista_lima_parte_2.csv",
    "/content/dataset_analista_lima_parte_3.csv"
]

CSV_FINAL = "dataset_analista_lima.csv"
lista_dfs = []

In [4]:
for f in files:
    if os.path.exists(f):
        print(f"Leyendo archivo: {f}")
        try:
            df_temp = pd.read_csv(f, encoding='utf-8')
            lista_dfs.append(df_temp)
        except Exception as e:
            print(f"Error al leer {f}: {e}. Omitiendo.")
    else:
        print(f"Archivo no encontrado: {f}.")

if lista_dfs:
    dataset_final_consolidado = pd.concat(lista_dfs, ignore_index=True)
    dataset_final_consolidado.to_csv(CSV_FINAL, index=False, encoding='utf-8')

    print("-" * 50)
    print(f"Total de registros: {len(dataset_final_consolidado)}")
    print(f"Archivo final guardado como: {CSV_FINAL}")
    print("-" * 50)
else:
    print("No se encontraron archivos válidos para consolidar.")

Leyendo archivo: /content/dataset_analista_lima_parte_1.csv
Leyendo archivo: /content/dataset_analista_lima_parte_2.csv
Leyendo archivo: /content/dataset_analista_lima_parte_3.csv
--------------------------------------------------
Total de registros: 4000
Archivo final guardado como: dataset_analista_lima.csv
--------------------------------------------------
